Visualización de Resultados de Ambigüedad Visual-Semántica

Este notebook genera las visualizaciones cualitativas para el artículo, explorando
cómo se distribuyen los cuadrantes de ambigüedad calculados al cruzar la similitud
semántica (Gower) y la similitud visual (OSNet).

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.manifold import TSNE

# Configuración de estética base
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams['figure.figsize'] = (10, 6)

out_dir_figs = '../../dataset/output/figures'
os.makedirs(out_dir_figs, exist_ok=True)

1. Carga de Datos
Cargaremos las tablas consolidadas del análisis cruzado y el resumen general poblacional.

In [ ]:
pairs_path = '../../dataset/output/metadata/gower_osnet_cross_analysis.csv'
summary_path = '../../dataset/output/metadata/gower_osnet_ambiguity_summary.csv'

pairs_df = pd.read_csv(pairs_path)
summary_df = pd.read_csv(summary_path)

# Extraer percentiles calculados
p90_gower = summary_df['p90_gower'].iloc[0]
p90_osnet = summary_df['p90_osnet'].iloc[0]

print(f"P90 Gower: {p90_gower:.4f}, P90 OSNet: {p90_osnet:.4f}")
print(f"Total de pares cargados: {len(pairs_df)}")

# Filtrar pares donde Gower u OSNet son nulos para gráficos principales
valid_df = pairs_df.dropna(subset=['gower_similarity', 'osnet_similarity']).copy()

# Orden de cuadrantes para colores
quadrant_order = [
    'baja_ambiguedad', 
    'semejanza_semantica_controlable', 
    'ambiguedad_visual_no_explicada', 
    'ambiguedad_confirmada'
]
quadrant_palette = {
    'baja_ambiguedad': '#cccccc',
    'semejanza_semantica_controlable': '#4c72b0', # Azul
    'ambiguedad_visual_no_explicada': '#dd8452', # Naranja
    'ambiguedad_confirmada': '#c44e52' # Rojo
}

2. Distribución de Cuadrantes de Ambigüedad
Frecuencias poblacionales de los pares.

In [ ]:
counts = valid_df['ambiguity_quadrant'].value_counts().reindex(quadrant_order)

plt.figure(figsize=(10, 5))
ax = sns.barplot(
    y=counts.index.str.replace('_', ' ').str.title(), 
    x=counts.values, 
    palette=list(quadrant_palette.values())
)
plt.title("Proporción de Cuadrantes de Ambigüedad")
plt.xlabel("Número de Pares")
plt.ylabel("")

# Añadir porcentajes
total = counts.sum()
for p in ax.patches:
    width = p.get_width()
    pct = 100 * width / total
    plt.text(width + total*0.01, p.get_y() + p.get_height()/2., 
             f"{int(width)} ({pct:.1f}%)", va='center')

plt.tight_layout()
plt.savefig(os.path.join(out_dir_figs, 'ambiguity_quadrants_bar.png'), dpi=300)
plt.show()

3. Dispersión Gower vs OSNet
Relación entre la semántica y la visual. Agregamos "jitter" en el eje X (Gower) porque es discreto.

In [ ]:
# Para visualizar mejor Gower, añadimos un leve jitter aleatorio
np.random.seed(42)
valid_df['gower_jitter'] = valid_df['gower_similarity'] + np.random.uniform(-0.02, 0.02, size=len(valid_df))

plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=valid_df,
    x='gower_jitter',
    y='osnet_similarity',
    hue='ambiguity_quadrant',
    palette=quadrant_palette,
    hue_order=quadrant_order,
    alpha=0.7,
    s=40,
    edgecolor=None
)

# Dibujar líneas de umbrales (P90)
plt.axvline(x=max(p90_gower, 0.80), color='black', linestyle='--', label=f'Gower Umbral ({max(p90_gower, 0.80):.2f})')
plt.axhline(y=p90_osnet, color='gray', linestyle='--', label=f'OSNet P90 ({p90_osnet:.4f})')

plt.title("Similitud de Atributos (Gower) vs Similitud Visual (OSNet)")
plt.xlabel("Similitud Semántica (Gower) + Jitter")
plt.ylabel("Similitud Visual Coseno (OSNet)")
plt.legend(title="Cuadrante", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(out_dir_figs, 'gower_vs_osnet_scatter.png'), dpi=300, bbox_inches='tight')
plt.show()

4. Contexto Adverso por Cuadrante
Verificamos empíricamente si la iluminación, oclusión y calidad degradada son mayores en pares visualmente ambiguos.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics = [
    ('pair_max_low_illum_ratio', 'Máxima Iluminación Baja'),
    ('pair_max_heavy_occlusion_ratio', 'Máxima Oclusión Severa'),
    ('pair_max_degraded_sil_ratio', 'Máxima Silueta Degradada')
]

for i, (col, title) in enumerate(metrics):
    sns.boxplot(
        data=valid_df, 
        x='ambiguity_quadrant', 
        y=col, 
        ax=axes[i],
        palette=quadrant_palette,
        order=quadrant_order
    )
    axes[i].set_title(title)
    axes[i].set_ylabel("Proporción (peor caso en el par)")
    axes[i].set_xlabel("")
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig(os.path.join(out_dir_figs, 'context_boxplots.png'), dpi=300)
plt.show()

5. Visualización Interactiva t-SNE de Secuencias
Proyección 2D del espacio latente de las secuencias para validar interactivamente la proximidad entre identidades confirmadas o no explicadas.

In [ ]:
# Cargar los vectores originales
osnet_seq_df = pd.read_csv('../../dataset/output/embeddings/sequence_embeddings_osnet.csv')
print("Parseando vectores JSON para t-SNE...")
osnet_seq_df['vector'] = osnet_seq_df['embedding_json'].apply(json.loads).apply(np.array)

# Reducir dimensionalidad con t-SNE
print("Calculando t-SNE (puede tomar un momento)...")
X = np.vstack(osnet_seq_df['vector'].values)
tsne = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto')
X_tsne = tsne.fit_transform(X)

osnet_seq_df['tsne_x'] = X_tsne[:, 0]
osnet_seq_df['tsne_y'] = X_tsne[:, 1]

# Identificar identidades que pertenecen a algún par "difícil" (para resaltarlas si se desea)
# Por ahora mostramos todas interactivamente.
# Convertimos ID a string para tratarlo como categoría discreta
osnet_seq_df['person_global_id_str'] = osnet_seq_df['person_global_id'].astype(str)

In [ ]:
fig = px.scatter(
    osnet_seq_df, 
    x='tsne_x', 
    y='tsne_y', 
    color='person_global_id_str', 
    hover_data=['sequence_id', 'cam_id', 'person_global_id'],
    title="t-SNE Interactivo de Secuencias (OSNet)",
    labels={"person_global_id_str": "Identidad"}
)

# Guardar HTML interactivo
html_out = os.path.join(out_dir_figs, 'tsne_sequences_interactive.html')
fig.write_html(html_out)
print(f"Gráfico t-SNE interactivo guardado en: {html_out}")

# Mostrar en el notebook
fig.show()

> **Nota**: El gráfico t-SNE generado se puede explorar interactivamente directamente aquí o abriendo `dataset/output/figures/tsne_sequences_interactive.html` en tu navegador.